In [5]:
# %%
import warnings
from pathlib import Path
import pickle

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from scipy.stats import pearsonr
from catboost import CatBoostRegressor

warnings.filterwarnings('ignore')

BASE_PATH = '/Users/egorilin/Desktop/COMA'
BASE = Path(BASE_PATH)

RANDOM_STATE = 42
TEST_SIZE = 0.2

CB_KW = dict(iterations=500, learning_rate=0.05, depth=6,
             verbose=False, random_seed=RANDOM_STATE)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')

TARGET_COLUMNS = [
    'S. aureus ATCC 43300 Activity MIC, мг/л',
    'S. aureus ATCC 43300 Activity MBC, мг/л',
    'E. coli ATCC 25922 Activity MIC, мг/л',
    'E. coli ATCC 25922 Activity MBC, мг/л',
]

# 'biocides' — 12 фич (без диполей); 'conc' — 15 фич (с диполями)
DATASET_MODE = 'conc'

FEATURE_COLS_FULL = [
    'Molecular weight', 'LogP', 'mmff94', 'gasteiger', 'eem2015bm',
    'Molformer_157', 'Molformer_197', 'Molformer_668', 'Molformer_85',
    'Molformer_127', 'Molformer_732', 'Molformer_543', 'Molformer_729',
    'Molformer_337', 'Molformer_686',
]
FEATURE_COLS_BIO = [
    'Molecular weight', 'LogP',
    'Molformer_157', 'Molformer_197', 'Molformer_668', 'Molformer_85',
    'Molformer_127', 'Molformer_732', 'Molformer_543', 'Molformer_729',
    'Molformer_337', 'Molformer_686',
]
FEATURE_COLS = FEATURE_COLS_BIO if DATASET_MODE == 'biocides' else FEATURE_COLS_FULL
DIPOLE_COLS = ['mmff94', 'gasteiger', 'eem2015bm']

DESC_OUT_DIR = BASE / 'descriptors_test_predictions'
EMB_OUT_DIR  = BASE / 'embeddings_test_predictions'
DESC_OUT_DIR.mkdir(exist_ok=True)
EMB_OUT_DIR.mkdir(exist_ok=True)

print(f'Dataset mode: {DATASET_MODE}  |  {len(FEATURE_COLS)} фич')
print(f'Output dirs:\n  {DESC_OUT_DIR}\n  {EMB_OUT_DIR}')

Device: cpu
Dataset mode: conc  |  15 фич
Output dirs:
  /Users/egorilin/Desktop/COMA/descriptors_test_predictions
  /Users/egorilin/Desktop/COMA/embeddings_test_predictions


In [12]:
target

,Unnamed: 0,General formula,Molecular weight,Gram_aureus,Gram_coli,LogP,"S. aureus ATCC 43300 Activity MIC, мг/л","S. aureus ATCC 43300 Activity MBC, мг/л","E. coli ATCC 25922 Activity MIC, мг/л","E. coli ATCC 25922 Activity MBC, мг/л"
0,0,CCCCCCCCCC[N+]1=CC=CC(OC2=CC=C(OC3=C[N+](CCCCC...,706.7,0,1,2.634,1.0,4.0,2.0,4.0
1,1,CCCCCCCCCCC[N+]1=CC=CC(OC2=CC=C(OC3=C[N+](CCCC...,734.7,0,1,3.523,4.0,4.0,500.0,500.0
2,2,CCCCCCCCCCCC[N+]1=CC=CC(OC2=CC=C(OC3=C[N+](CCC...,762.8,0,1,3.810,32.0,125.0,8.0,8.0
3,3,CCCCCCCCCCC[N+](C=C1)=CC=C1OC2=CC=C(OC3=CC=[N+...,734.7,0,1,3.523,16.0,32.0,16.0,16.0
4,4,CCCCCCCCCC[N+](C=CC=C1)=C1OC2=CC=C(OC3=[N+](CC...,706.7,0,1,2.634,2.0,8.0,4.0,4.0
...,...,...,...,...,...,...,...,...,...,...
103,103,CC(C1=CC=C(OC2=[N+](CCCCCC)C=CC=C2)C=C1)(C)C3=...,806.6,0,1,1.935,0.5,1.0,8.0,16.0
104,104,CC(C1=CC=C(OC2=[N+](CCCCCCC)C=CC=C2)C=C1)(C)C3...,834.7,0,1,2.824,2.0,2.0,4.0,4.0
105,105,CC(C1=CC=C(OC2=[N+](CCCCCCCC)C=CC=C2)C=C1)(C)C...,862.7,0,1,3.714,4.0,4.0,4.0,4.0
106,106,CC(C1=CC=C(OC2=[N+](CCCCCCCCCC)C=CC=C2)C=C1)(C...,918.8,0,1,5.492,0.5,0.5,32.0,125.0


In [14]:
# %%
target = pd.read_excel(f'{BASE_PATH}/Data_biocides.xlsx')
desc_df = pd.read_excel(f'{BASE_PATH}/Data_prepare_biocides_molformer_new_embeddings.xlsx')
data_drop = pd.read_excel(f'{BASE_PATH}/Test_molformer_original_biocides.xlsx').drop(
    ['Unnamed: 0'], axis=1
)
dipoles_df = pd.read_csv(f'{BASE_PATH}/Train_dipole_moments.csv')

print(f'target:     {target.shape}')
print(f'desc_df:    {desc_df.shape}')
print(f'data_drop:  {data_drop.shape}')
print(f'dipoles_df: {dipoles_df.shape}  | cols: {list(dipoles_df.columns)}')

# --- Базовые фичи (LogP, Molecular weight) берём из target по позиции ---
BASE_FEATURE_COLS = ['Molecular weight', 'LogP']
missing_in_target = [c for c in BASE_FEATURE_COLS if c not in target.columns]
if missing_in_target:
    raise ValueError(f'В target нет колонок: {missing_in_target}. '
                     f'Колонки target: {list(target.columns)[:20]}')

desc_df = desc_df.reset_index(drop=True)
target_reset = target.reset_index(drop=True)

# выкидываем эти колонки из desc_df, если уже есть, и подставляем из target по позиции
desc_df = desc_df.drop(columns=[c for c in BASE_FEATURE_COLS if c in desc_df.columns])
n_base = min(len(desc_df), len(target_reset))
if len(desc_df) != len(target_reset):
    print(f'  ⚠️  Длины desc_df({len(desc_df)}) и target({len(target_reset)}) не совпадают, '
          f'возьму первые {n_base}')

for c in BASE_FEATURE_COLS:
    desc_df[c] = np.nan
    desc_df.loc[:n_base - 1, c] = target_reset[c].iloc[:n_base].values

print('\nПодставлено из target:')
for c in BASE_FEATURE_COLS:
    n_nan = desc_df[c].isna().sum()
    print(f'  {c}: {len(desc_df) - n_nan}/{len(desc_df)} заполнено  ({n_nan} NaN)')

# --- Подстановка диполей по позиции из csv ---
first_col = dipoles_df.columns[0]
if dipoles_df[first_col].dtype.kind in 'iu' or first_col.lower() in ('unnamed: 0', 'index', 'id', '№'):
    print(f'\n  Отбрасываю служебную первую колонку csv: "{first_col}"')
    dipoles_df = dipoles_df.drop(columns=[first_col])

have_dipoles = [c for c in DIPOLE_COLS if c in dipoles_df.columns]
miss_dipoles = [c for c in DIPOLE_COLS if c not in dipoles_df.columns]
print(f'  Диполи в csv: {have_dipoles}')
if miss_dipoles:
    print(f'  ⚠️  Нет в csv диполей: {miss_dipoles}')

n_dip = min(len(desc_df), len(dipoles_df))
if len(dipoles_df) != len(desc_df):
    print(f'  ⚠️  Длины desc_df({len(desc_df)}) и dipoles_df({len(dipoles_df)}) не совпадают, '
          f'возьму первые {n_dip}')

desc_df = desc_df.drop(columns=[c for c in DIPOLE_COLS if c in desc_df.columns])
dipoles_aligned = dipoles_df[have_dipoles].reset_index(drop=True)

for c in have_dipoles:
    desc_df[c] = np.nan
    desc_df.loc[:n_dip - 1, c] = dipoles_aligned[c].iloc[:n_dip].values

print('\nПокрытие диполей после подстановки:')
for c in have_dipoles:
    n_nan = desc_df[c].isna().sum()
    print(f'  {c}: {len(desc_df) - n_nan}/{len(desc_df)} заполнено  ({n_nan} NaN)')

# --- Таргеты тоже должны быть в desc_df для Ячейки 4. Если их там нет — берём из target по позиции ---
for c in TARGET_COLUMNS:
    if c not in desc_df.columns:
        if c not in target_reset.columns:
            raise ValueError(f'Нет таргета "{c}" ни в desc_df, ни в target')
        desc_df[c] = np.nan
        n_tg = min(len(desc_df), len(target_reset))
        desc_df.loc[:n_tg - 1, c] = target_reset[c].iloc[:n_tg].values

# --- Финальные проверки ---
missing_desc = [c for c in FEATURE_COLS if c not in desc_df.columns]
if missing_desc:
    raise ValueError(f'Нет колонок в desc_df после всех подстановок: {missing_desc}')

for c in TARGET_COLUMNS:
    if c not in target.columns:
        raise ValueError(f'Нет таргета в target: {c}')

print('\n✓ Все нужные колонки на месте')
print(f'  FEATURE_COLS ({len(FEATURE_COLS)}): {FEATURE_COLS}')

target:     (108, 10)
desc_df:    (108, 769)
data_drop:  (108, 768)
dipoles_df: (108, 4)  | cols: ['Unnamed: 0', 'mmff94', 'gasteiger', 'eem2015bm']

Подставлено из target:
  Molecular weight: 108/108 заполнено  (0 NaN)
  LogP: 101/108 заполнено  (7 NaN)

  Отбрасываю служебную первую колонку csv: "Unnamed: 0"
  Диполи в csv: ['mmff94', 'gasteiger', 'eem2015bm']

Покрытие диполей после подстановки:
  mmff94: 108/108 заполнено  (0 NaN)
  gasteiger: 108/108 заполнено  (0 NaN)
  eem2015bm: 108/108 заполнено  (0 NaN)

✓ Все нужные колонки на месте
  FEATURE_COLS (15): ['Molecular weight', 'LogP', 'mmff94', 'gasteiger', 'eem2015bm', 'Molformer_157', 'Molformer_197', 'Molformer_668', 'Molformer_85', 'Molformer_127', 'Molformer_732', 'Molformer_543', 'Molformer_729', 'Molformer_337', 'Molformer_686']


In [ ]:
# %%
class AdvancedContrastiveEncoder(nn.Module):
    def __init__(self, input_dim, hidden_dims=[512, 384, 128],
                 projection_dim=, dropout=0.3):
        super().__init__()
        layers = []
        prev_dim = input_dim
        for hidden_dim in hidden_dims:
            layers.extend([
                nn.Linear(prev_dim, hidden_dim),
                nn.BatchNorm1d(hidden_dim),
                nn.ReLU(),
                nn.Dropout(dropout),
            ])
            prev_dim = hidden_dim
        self.encoder = nn.Sequential(*layers)
        self.projection_head = nn.Sequential(
            nn.Linear(hidden_dims[-1], projection_dim),
            nn.BatchNorm1d(projection_dim),
            nn.ReLU(),
            nn.Dropout(dropout * 0.5),
            nn.Linear(projection_dim, projection_dim),
            nn.BatchNorm1d(projection_dim),
        )

    def forward(self, x, return_projection=True):
        features = self.encoder(x)
        if return_projection:
            return features, self.projection_head(features)
        return features


encoder = AdvancedContrastiveEncoder(
    input_dim=768, hidden_dims=[512, 384, 128],
    projection_dim=128, dropout=0.3,
).to(device)
encoder.load_state_dict(
    torch.load(f'{BASE_PATH}/encoder_borderline_smote.pth', map_location=device)
)
encoder.eval()

with open(f'{BASE_PATH}/scaler_classification.pkl', 'rb') as f:
    scaler = pickle.load(f)

data_scaled = scaler.transform(data_drop)
X_tensor = torch.FloatTensor(data_scaled).to(device)
with torch.no_grad():
    molecular_embeddings = encoder(X_tensor, return_projection=False).cpu().numpy()

print(f'✓ molecular_embeddings: {molecular_embeddings.shape}')

RuntimeError: Error(s) in loading state_dict for AdvancedContrastiveEncoder:
	size mismatch for encoder.8.weight: copying a param with shape torch.Size([256, 384]) from checkpoint, the shape in current model is torch.Size([128, 384]).
	size mismatch for encoder.8.bias: copying a param with shape torch.Size([256]) from checkpoint, the shape in current model is torch.Size([128]).
	size mismatch for encoder.9.weight: copying a param with shape torch.Size([256]) from checkpoint, the shape in current model is torch.Size([128]).
	size mismatch for encoder.9.bias: copying a param with shape torch.Size([256]) from checkpoint, the shape in current model is torch.Size([128]).
	size mismatch for encoder.9.running_mean: copying a param with shape torch.Size([256]) from checkpoint, the shape in current model is torch.Size([128]).
	size mismatch for encoder.9.running_var: copying a param with shape torch.Size([256]) from checkpoint, the shape in current model is torch.Size([128]).
	size mismatch for